Використай цей шаблон в роботі з датасетом.      
Ти можеш додавати комірки за потреби, але не змінюй структуру і послідовність питань.      
Обмежся функціями з наведених бібліотек.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

1. Завантаж датасет з бібліотеки seaborn:

In [15]:
df = sns.load_dataset('titanic')

2. Переглянь перші рядки датасету. Зроби висновок, чи коректно він завантажився.

In [5]:
df.head(10)
#Датасет завантажився коркентно, можна приступати до аналізу даних.

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


3. Перевір типи стовпців. Які з них потребують перетворення?

In [6]:
df.info()

# 1. Стовпці 'sex' та 'embarked' мають тип object — їх варто перетворити на category для економії пам'яті.
# 2. Стовпець 'survived' можна перетворити з int64 на bool, оскільки він містить лише значення 0 та 1.
# 3. Стовпець 'age' має тип float64 та містить пропуски (714 із 891) — потребує обробки NaN. Також можлтво варто розглянути перетворення на int після заповнення пропусків.
# 4. Стовпець 'deck' має критичну кількість пропущених значень (лише 203 non-null).

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


4. Перевір статистику по УСІМ стовпцям датасету.

In [7]:
df.describe(include='all')

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
count,891.000000,891.000000,891,714.000000,891.000000,891.000000,891.000000,889,891,891,891,203,889,891,891
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,3,3,3,2,7,3,2,2
top,NaN,NaN,male,NaN,NaN,NaN,NaN,S,Third,man,True,C,Southampton,no,True
freq,NaN,NaN,577,NaN,NaN,NaN,NaN,644,491,537,537,59,644,549,537
mean,0.383838,2.308642,NaN,29.699118,0.523008,0.381594,32.204208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.486592,0.836071,NaN,14.526497,1.102743,0.806057,49.693429,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,0.000000,2.000000,NaN,20.125000,0.000000,0.000000,7.910400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.000000,3.000000,NaN,28.000000,0.000000,0.000000,14.454200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.000000,3.000000,NaN,38.000000,1.000000,0.000000,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


5. Необхідно створити єдиний стовпчик, що вказує кількість родичів для кожного пасажира на борту, замість:
- кількість братів/сестер або чоловіків/дружин на борту;
- кількість батьків або дітей на борту;

Булева ознака: стовпчик 'alone'=True, якщо пасажир подорожував один (без родичів на борту).        
Перевір, щоб для одиноких пасажирів новий стовпчик мав значення 0.      
Після створення нового стовпчика, дропни "sibsp", "parch", "alone".

In [17]:
# Створюємо нову колонку
df['relatives'] = df['sibsp'] + df['parch']

# Видаляємо непотрібні колонки
# df = df.drop(columns=['sibsp', 'parch', 'alone'])

df.loc[df['alone']==True,'relatives']=0
df = df.drop(columns=['sibsp', 'parch', 'alone'])
a=sum(df['relatives'].isna())
print(a)
df.head(20)

0


,survived,pclass,sex,age,fare,embarked,class,who,adult_male,deck,embark_town,alive,relatives
0,0,3,male,22.0,7.2500,S,Third,man,True,NaN,Southampton,no,1
1,1,1,female,38.0,71.2833,C,First,woman,False,C,Cherbourg,yes,1
2,1,3,female,26.0,7.9250,S,Third,woman,False,NaN,Southampton,yes,0
3,1,1,female,35.0,53.1000,S,First,woman,False,C,Southampton,yes,1
4,0,3,male,35.0,8.0500,S,Third,man,True,NaN,Southampton,no,0
5,0,3,male,NaN,8.4583,Q,Third,man,True,NaN,Queenstown,no,0
6,0,1,male,54.0,51.8625,S,First,man,True,E,Southampton,no,0
7,0,3,male,2.0,21.0750,S,Third,child,False,NaN,Southampton,no,4
8,1,3,female,27.0,11.1333,S,Third,woman,False,NaN,Southampton,yes,2
9,1,2,female,14.0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,1


6. Перевір, як часто зустрічається певна кількість родичів в новому стовпці.    
Результат представ у вигляді таблиці, побудованої з використанням групування та агрегації:

In [137]:
relatives_stats = df.groupby('relatives').size()
relatives_stats = relatives_stats.reset_index(name='count')

print(relatives_stats)

   relatives  count
0          0    537
1          1    161
2          2    102
3          3     29
4          4     15
5          5     22
6          6     12
7          7      6
8         10      7


7. Використовуючи цикл заміни кількість родичів, що перевищує число 5(п'ять) на значення "above 5"(понад п'ять).       
Запиши значення в новий стовпчик ʼrelatives_categoryʼ.

In [138]:
relatives_category = []

for count in df['relatives']:
    if count > 5:
        relatives_category.append('above 5')
    else:
        relatives_category.append(count)
        
df['relatives_category'] = relatives_category

df.head(20)

,survived,pclass,sex,age,fare,embarked,class,who,adult_male,deck,embark_town,alive,relatives,relatives_category
0,0,3,male,22.0,7.2500,S,Third,man,True,NaN,Southampton,no,1,1
1,1,1,female,38.0,71.2833,C,First,woman,False,C,Cherbourg,yes,1,1
2,1,3,female,26.0,7.9250,S,Third,woman,False,NaN,Southampton,yes,0,0
3,1,1,female,35.0,53.1000,S,First,woman,False,C,Southampton,yes,1,1
4,0,3,male,35.0,8.0500,S,Third,man,True,NaN,Southampton,no,0,0
5,0,3,male,NaN,8.4583,Q,Third,man,True,NaN,Queenstown,no,0,0
6,0,1,male,54.0,51.8625,S,First,man,True,E,Southampton,no,0,0
7,0,3,male,2.0,21.0750,S,Third,child,False,NaN,Southampton,no,4,4
8,1,3,female,27.0,11.1333,S,Third,woman,False,NaN,Southampton,yes,2,2
9,1,2,female,14.0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,1,1


8. Необхідно вивести на екран статистику по відсотковому показнику пасажирів з кількістю родичів більше 5 відносно загального числа пасажирів для кожного з міст посадки. Для цього:
- порахуй загальну кількість пасажирів в кожному з міст посадки,
- знайди число пасажирів з кількістю родичів більше 5 в кожному з міст посадки,
- поділи ці два стовчики між собою, перетворивши результат у відсотки (ціле число).        
Відобрази статистику для усіх міст посадки.

In [139]:
total_by_city = df.groupby('embarked').size()

passengers_above_5 = df[df['relatives_category'] == 'above 5']

grouped_by_city = passengers_above_5.groupby('embarked')

above_5_by_city = grouped_by_city.size()

percentage_stats = (above_5_by_city / total_by_city * 100).fillna(0).astype(int)
print(percentage_stats)


embarked
C    0
Q    0
S    3
dtype: int64


9. Заповни відсутні значення віку медіаною.

In [140]:
age_median = df['age'].median()

df['age'] = df['age'].fillna(age_median)

df.head(20)

,survived,pclass,sex,age,fare,embarked,class,who,adult_male,deck,embark_town,alive,relatives,relatives_category
0,0,3,male,22.0,7.2500,S,Third,man,True,NaN,Southampton,no,1,1
1,1,1,female,38.0,71.2833,C,First,woman,False,C,Cherbourg,yes,1,1
2,1,3,female,26.0,7.9250,S,Third,woman,False,NaN,Southampton,yes,0,0
3,1,1,female,35.0,53.1000,S,First,woman,False,C,Southampton,yes,1,1
4,0,3,male,35.0,8.0500,S,Third,man,True,NaN,Southampton,no,0,0
5,0,3,male,28.0,8.4583,Q,Third,man,True,NaN,Queenstown,no,0,0
6,0,1,male,54.0,51.8625,S,First,man,True,E,Southampton,no,0,0
7,0,3,male,2.0,21.0750,S,Third,child,False,NaN,Southampton,no,4,4
8,1,3,female,27.0,11.1333,S,Third,woman,False,NaN,Southampton,yes,2,2
9,1,2,female,14.0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,1,1


10. Створи новий стовпець, де вік представлено категорією, замість числа (наприклад: до 14 років, 14-34 роки, 35-59 років, 60 і більше років). Виконай задачу з використанням користувацької функції. Осіб з невідомим віком познач відповідно.

In [141]:
#Створюємо функцію для розподілу на категорії
def categorize_age(age):
    if age < 14:
        return 'до 14 років'
    elif age < 35:
        return '14-34 роки'
    elif age < 60:
        return '35-59 років'
    else:
        return '60 і більше років'


df['age_category'] = df['age'].apply(categorize_age) # Додаємо нову категоріальну колонку на основі віку

df.head(20)

,survived,pclass,sex,age,fare,embarked,class,who,adult_male,deck,embark_town,alive,relatives,relatives_category,age_category
0,0,3,male,22.0,7.2500,S,Third,man,True,NaN,Southampton,no,1,1,14-34 роки
1,1,1,female,38.0,71.2833,C,First,woman,False,C,Cherbourg,yes,1,1,35-59 років
2,1,3,female,26.0,7.9250,S,Third,woman,False,NaN,Southampton,yes,0,0,14-34 роки
3,1,1,female,35.0,53.1000,S,First,woman,False,C,Southampton,yes,1,1,35-59 років
4,0,3,male,35.0,8.0500,S,Third,man,True,NaN,Southampton,no,0,0,35-59 років
5,0,3,male,28.0,8.4583,Q,Third,man,True,NaN,Queenstown,no,0,0,14-34 роки
6,0,1,male,54.0,51.8625,S,First,man,True,E,Southampton,no,0,0,35-59 років
7,0,3,male,2.0,21.0750,S,Third,child,False,NaN,Southampton,no,4,4,до 14 років
8,1,3,female,27.0,11.1333,S,Third,woman,False,NaN,Southampton,yes,2,2,14-34 роки
9,1,2,female,14.0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,1,1,14-34 роки


11. Перевір, в якій віковій категорії була найвища смертність.     
Для цього рекомендується перетворити стовпець 'alive' в булевий тип.    
 Потім підрахувати загальну кількість пасажирів та кількість тих, хто не вижив.      
 Потім обчисли відносний показниках для кожної категорії.

In [142]:
# Створюємо нову колонку: True для тих, хто загинув, False для тих, хто вижив
df['is_dead'] = df['alive'] == 'no'

# Групуємо дані за категоріями віку. Для кожної категорії рахуємо загальну кількість людей та кількість загиблих.
stats = df.groupby('age_category')['is_dead'].agg(['count', 'sum'])


#Обчислюємо відсоток смертності 
stats['rate_die'] = (stats['sum'] / stats['count'] * 100).astype(int)

print(stats)

                   count  sum  rate_die
age_category                           
14-34 роки           585  379        64
35-59 років          209  122        58
60 і більше років     26   19        73
до 14 років           71   29        40


*Бонусне завдання*         :          
Підготуй розгорнуту статистику смертності по категорії віку, класу квитка, рівню каюти та кількості родичів.      
Які фактори, на твою думку, найбільше пов'язані з рівнем смерності? (наприклад: найбільша смертність у відсотковому значенні спостерігається серед вікової групи ... класу квитка.... при наявності ... родичів та для рівня каюти.... Фактор ... має найвищий вплив на смертність)

In [143]:
# 1. Готуємо дані (робимо все текстом)
for col in ['age_category', 'pclass', 'deck', 'relatives_category']:
    df[col] = df[col].astype(str)

# 2. Створюємо функцію
def show_titanic_stat(column_name, title):
    print(f"--- {title} ---")
    # Рахуємо статистику
    res = df.groupby(column_name)['is_dead'].agg(['count', 'mean'])
    # Робимо відсотки цілими числами
    res['mean'] = (res['mean'] * 100).astype(int)
    # Змінюємо назви для зручності
    res.columns = ['Всього пасажирів', '% Смертності']
    # Виводимо результат, відсортований від найбільшої смертності
    print(res.sort_values(by='% Смертності', ascending=False))
    print("\n")

In [144]:
show_titanic_stat('age_category', 'СТАТИСТИКА ЗА КАТЕГОРІЯМИ ВІКУ')
show_titanic_stat('pclass', 'СТАТИСТИКА ЗА КЛАСОМ КВИТКА')

--- СТАТИСТИКА ЗА КАТЕГОРІЯМИ ВІКУ ---
                   Всього пасажирів  % Смертності
age_category                                     
60 і більше років                26            73
14-34 роки                      585            64
35-59 років                     209            58
до 14 років                      71            40


--- СТАТИСТИКА ЗА КЛАСОМ КВИТКА ---
        Всього пасажирів  % Смертності
pclass                                
3                    491            75
2                    184            52
1                    216            37




In [145]:
show_titanic_stat('deck', 'СТАТИСТИКА ЗА ПАЛУБОЮ (DECK)')
show_titanic_stat('relatives_category', 'СТАТИСТИКА ЗА РОДИЧАМИ')

--- СТАТИСТИКА ЗА ПАЛУБОЮ (DECK) ---
      Всього пасажирів  % Смертності
deck                                
nan                688            70
A                   15            53
G                    4            50
C                   59            40
F                   13            38
B                   47            25
E                   32            25
D                   33            24


--- СТАТИСТИКА ЗА РОДИЧАМИ ---
                    Всього пасажирів  % Смертності
relatives_category                                
5                                 22            86
above 5                           25            84
4                                 15            80
0                                537            69
1                                161            44
2                                102            42
3                                 29            27




Вікова група: Найбільша смертність спостерігається серед пасажирів віком 60 і більше років (73%). Це вказує на те, що літнім людям було фізично найважче евакуюватися в екстремальних умовах.

Клас квитка: Найвищий рівень смертності зафіксовано у 3 класі (75%). Для порівняння, у 1 класі цей показник становить лише 37%.

Кількість родичів: Найменші шанси на виживання мали пасажири з великими сім'ями (категорія 5 родичів — 86% та above 5 — 84%). Також висока смертність серед одинаків (69%).

Рівень каюти (Deck): Найвищий відсоток загиблих серед тих, чия палуба не була вказана (nan — 70%), а також на нижніх палубах та палубі А.

Визначальним фактором став клас квитка (pclass), оскільки він визначав пріоритет при посадці в шлюпки та фізичну близькість каюти до верхньої палуби. Це призвело до того, що пасажири 3-го класу гинули вдвічі частіше, ніж пасажири 1-го класу.